# Phase 3: MLP Neural Network — Flight Delay Prediction

**Group 01-02 | DS261 Spring 2026**

This notebook implements a **Multilayer Perceptron (MLP)** neural network for binary
classification of flight departure delays (DEP_DEL15). It uses a **bespoke preprocessing
pipeline** distinct from the existing MinMax/AssemblerOnly paths:

- **StandardScaler** (z-score normalization) instead of MinMax — better gradient flow,
  more robust to outliers, centers data for sigmoid activations
- **PCA** (optional) for dimensionality reduction — reduces multicollinearity among the
  99 engineered features

**Architectures explored** (minimum 2 required by rubric):
- MLP-A: `[N, 64, 2]` — shallow baseline (1 hidden layer)
- MLP-B: `[N, 128, 64, 2]` — deeper (2 hidden layers)
- MLP-C: `[N, 256, 128, 2]` — wider first layer

**Key design differences from LR/RF/GBT notebooks:**
- StandardScaler preprocessing (not MinMax)
- PCA experiment for feature reduction
- MLP has no `weightCol` → uses 2:1 undersampling
- Pseudo-learning curves via incremental `maxIter`
- **Scikit-learn ablation study**: Adam optimizer, L2 regularization, convergence
  curves, and feature family ablation (complements PySpark's limited MLP API)

See also: `phase2.5_modeling_MinMax_LR_RF`, `phase2.5_modeling_GBT` for the other models.

In [0]:
import json
import time
import os
from io import StringIO

import pyspark.sql.functions as F
from pyspark.sql.types import *
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from pyspark.ml.classification import MultilayerPerceptronClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import VectorAssembler, StandardScaler, PCA
from pyspark.ml import Pipeline
from pyspark.ml.functions import vector_to_array

# ── Configuration ──────────────────────────────────────────────────────────────
BASE_GROUP     = "dbfs:/student-groups/Group_01_02"
TARGET         = "DEP_DEL15"
BETA           = 2.0   # F-β weight: recall valued 2× over precision
SEED           = 42

# Phase 3.3 feature-engineered parquets (pre-assembly)
TRAIN_FE_PATH  = f"{BASE_GROUP}/df_train_phase3.parquet"
VAL_FE_PATH    = f"{BASE_GROUP}/df_val_phase3.parquet"
TEST_FE_PATH   = f"{BASE_GROUP}/df_test_phase3.parquet"
NUM_COLS_PATH  = f"{BASE_GROUP}/phase3_final_num_cols.json"

# Output paths
MLP_RESULTS_CSV   = f"{BASE_GROUP}/phase3_results_summary_mlp.csv"
CHART_DIR          = "/dbfs/FileStore/group_01_02/phase3_charts"

# Undersampling ratio (on-time : delayed)
UNDERSAMPLE_RATIO = 2.0

# PCA: retain components explaining this fraction of variance
PCA_VARIANCE_TARGET = 0.95

print("Libraries loaded (MLP modeling notebook).")
print(f"F-β metric: β = {BETA}  (recall weighted {int(BETA)}× over precision)")

Libraries loaded (MLP modeling notebook).
F-β metric: β = 2.0  (recall weighted 2× over precision)


## 1. Data Loading

Load Phase 3.3 feature-engineered parquets and the ordered feature manifest.
These parquets contain all 99 numeric features as individual columns (not yet assembled
into a vector), plus `DEP_DEL15` and metadata columns.

In [0]:
# Load feature manifest
num_cols_raw = dbutils.fs.head(NUM_COLS_PATH, 100000)
numerical_cols = json.loads(num_cols_raw)
N_FEATURES = len(numerical_cols)
print(f"Feature manifest: {N_FEATURES} features from phase3_final_num_cols.json")
print(f"First 10: {numerical_cols[:10]}")
print(f"Last 5:   {numerical_cols[-5:]}")

Feature manifest: 104 features from phase3_final_num_cols.json
First 10: ['QUARTER', 'MONTH', 'DAY_OF_MONTH', 'YEAR', 'DISTANCE', 'CRS_DEP_TIME', 'dep_hour', 'is_morning_peak', 'is_evening_peak', 'is_weekend']
Last 5:   ['late_night_x_precip_scl', 'late_night_x_wind_scl', 'late_night_x_vis_bin', 'prev2h_busy_x_precip_scl', 'prev2h_busy_x_wind_scl']


In [0]:
df_train_fe = spark.read.parquet(TRAIN_FE_PATH).cache()
df_val_fe   = spark.read.parquet(VAL_FE_PATH).cache()
df_test_fe  = spark.read.parquet(TEST_FE_PATH).cache()

# Verify all expected columns exist
missing = [c for c in numerical_cols if c not in df_train_fe.columns]
if missing:
    print(f"WARNING: {len(missing)} columns in manifest not found on parquet: {missing[:10]}")
    numerical_cols = [c for c in numerical_cols if c in df_train_fe.columns]
    N_FEATURES = len(numerical_cols)
    print(f"Adjusted to {N_FEATURES} features.")
else:
    print(f"All {N_FEATURES} manifest columns present on train parquet.")

n_train = df_train_fe.count()
print(f"Train rows: {n_train:,}")

All 104 manifest columns present on train parquet.
Train rows: 18,698,999


## 2. Preprocessing Pipeline — StandardScaler + PCA

**Why StandardScaler over MinMax?**
- Z-score normalization (mean=0, std=1) is generally preferred for neural networks
- More robust to outliers than MinMax (which squashes everything to [0,1])
- Better gradient flow through sigmoid activations when inputs are centered
- StandardScaler is fit on **train only** and applied to val/test (no leakage)

**Why PCA?**
- The 99 features include many correlated interaction terms
- PCA reduces multicollinearity and can improve training stability
- We experiment with both full-feature and PCA-reduced MLP to compare
- PCA is fit on **train only** (no leakage)

In [0]:
# ── 2a. StandardScaler Pipeline ───────────────────────────────────────────────
#
# NOTE: PySpark StandardScaler with withMean=True requires DENSE vectors.
# VectorAssembler may produce SparseVector if many values are zero.
# We force dense conversion between assembly and scaling to avoid a runtime crash.

from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.sql.functions import udf

to_dense_udf = udf(lambda v: Vectors.dense(v.toArray()) if v is not None else None, VectorUDT())

# Null audit: any null in the feature columns will cause silent row drops.
# Phase 3.3 should have imputed all nulls; assert here so failures are visible.
_null_counts = {c: df_train_fe.filter(F.col(c).isNull()).count() for c in numerical_cols}
_cols_with_nulls = {c: n for c, n in _null_counts.items() if n > 0}
if _cols_with_nulls:
    print(f"WARNING: {len(_cols_with_nulls)} feature column(s) have nulls — re-run Phase 3.3 imputation:")
    for c, n in sorted(_cols_with_nulls.items(), key=lambda x: -x[1])[:10]:
        print(f"  {c}: {n:,} nulls")
else:
    print("Null audit passed: no nulls in feature columns.")

assembler = VectorAssembler(
    inputCols=numerical_cols,
    outputCol="features_sparse",
    handleInvalid="error",   # fail loudly if nulls reach assembly after imputation
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True,
)

print("Fitting StandardScaler pipeline on train...")
t0 = time.time()

# Step 1: Assemble features
assembler_model = assembler  # VectorAssembler is not an Estimator, no fit needed

def assemble_and_densify(df):
    """Assemble features and convert to dense vector for StandardScaler compatibility."""
    return assembler.transform(df).withColumn("features_raw", to_dense_udf("features_sparse")).drop("features_sparse")

df_train_assembled = assemble_and_densify(df_train_fe).cache()

# Step 2: Fit StandardScaler on train
scaler_model = scaler.fit(df_train_assembled)
std_fit_time = time.time() - t0
print(f"StandardScaler fit done in {std_fit_time:.1f}s")

# Step 3: Transform all splits
# Carry a temporal sort key for time-series CV (YEAR*10000 + MONTH*100 + DAY_OF_MONTH)
TEMPORAL_COL = "_temporal_sort"

def scale_split(df, keep_temporal=False):
    assembled = assemble_and_densify(df)
    scaled = scaler_model.transform(assembled)
    if keep_temporal and "YEAR" in df.columns and "MONTH" in df.columns and "DAY_OF_MONTH" in df.columns:
        scaled = scaled.withColumn(
            TEMPORAL_COL,
            (F.col("YEAR") * 10000 + F.col("MONTH") * 100 + F.col("DAY_OF_MONTH")).cast("int")
        )
        return scaled.select("features", TARGET, TEMPORAL_COL)
    return scaled.select("features", TARGET)

# Train keeps temporal column for CV; val/test don't need it
df_train_std = scale_split(df_train_fe, keep_temporal=True).cache()
df_val_std   = scale_split(df_val_fe).cache()
df_test_std  = scale_split(df_test_fe).cache()

df_train_assembled.unpersist()

_dim = df_train_std.first()["features"].size
print(f"Feature vector size (StandardScaler): {_dim}")

Null audit passed: no nulls in feature columns.
Fitting StandardScaler pipeline on train...
StandardScaler fit done in 1171.0s
Feature vector size (StandardScaler): 104


In [0]:
# ── 2b. PCA Pipeline ─────────────────────────────────────────────────────────
# Fit PCA on the StandardScaled train features to find optimal component count

# First, fit PCA with all components to analyze variance
pca_full = PCA(k=N_FEATURES, inputCol="features", outputCol="pca_features_full")
pca_full_model = pca_full.fit(df_train_std)

explained_variance = pca_full_model.explainedVariance.toArray()
cumulative_variance = np.cumsum(explained_variance)

# Find k for target variance
k_target = int(np.searchsorted(cumulative_variance, PCA_VARIANCE_TARGET) + 1)
k_target = min(k_target, N_FEATURES)

print(f"PCA Variance Analysis:")
print(f"  Total components: {N_FEATURES}")
print(f"  Components for {PCA_VARIANCE_TARGET*100:.0f}% variance: {k_target}")
print(f"  Variance at k={k_target}: {cumulative_variance[k_target-1]*100:.2f}%")
print(f"  Top 10 component variances: {[f'{v:.4f}' for v in explained_variance[:10]]}")

PCA Variance Analysis:
  Total components: 104
  Components for 95% variance: 55
  Variance at k=55: 95.19%
  Top 10 component variances: ['0.0901', '0.0747', '0.0656', '0.0453', '0.0388', '0.0357', '0.0346', '0.0315', '0.0295', '0.0273']


In [0]:
# Save PCA variance explained chart
os.makedirs(CHART_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, N_FEATURES + 1), cumulative_variance * 100, "b-", linewidth=2)
ax.axhline(y=PCA_VARIANCE_TARGET * 100, color="r", linestyle="--",
           label=f"{PCA_VARIANCE_TARGET*100:.0f}% threshold")
ax.axvline(x=k_target, color="g", linestyle="--",
           label=f"k={k_target} components")
ax.set_xlabel("Number of Principal Components", fontsize=12)
ax.set_ylabel("Cumulative Explained Variance (%)", fontsize=12)
ax.set_title("PCA: Cumulative Explained Variance", fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(1, N_FEATURES)
ax.set_ylim(0, 102)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/pca_variance_explained.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/pca_variance_explained.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/pca_variance_explained.png


In [0]:
# Fit PCA with optimal k and transform
pca_reduced = PCA(k=k_target, inputCol="features", outputCol="pca_features")
pca_model = pca_reduced.fit(df_train_std)

df_train_pca = (pca_model.transform(df_train_std)
                .drop("features")
                .withColumnRenamed("pca_features", "features")
                .cache())
df_val_pca   = (pca_model.transform(df_val_std)
                .drop("features")
                .withColumnRenamed("pca_features", "features")
                .cache())
df_test_pca  = (pca_model.transform(df_test_std)
                .drop("features")
                .withColumnRenamed("pca_features", "features")
                .cache())

k_actual = df_train_pca.first()["features"].size
print(f"PCA-reduced feature vector size: {k_actual}")

PCA-reduced feature vector size: 55


## 3. Undersampling

PySpark's `MultilayerPerceptronClassifier` does **not** support a `weightCol` parameter
for class weighting. To handle the ~4:1 class imbalance, we undersample the majority
class (on-time flights) to a 2:1 ratio, consistent with the RF and GBT approaches.

Validation and test sets are **never** undersampled.

In [0]:
def undersample(df, target_col, ratio, seed=SEED):
    """Undersample majority class to achieve target on-time:delayed ratio."""
    n_delayed = df.filter(F.col(target_col) == 1).count()
    n_ontime  = df.filter(F.col(target_col) == 0).count()
    target_ontime = int(ratio * n_delayed)

    print(f"  Before: on-time={n_ontime:,}  delayed={n_delayed:,}  ratio={n_ontime/max(n_delayed,1):.2f}:1")

    if target_ontime < n_ontime:
        fraction = target_ontime / n_ontime
        df_ontime_sample  = df.filter(F.col(target_col) == 0).sample(fraction=fraction, seed=seed)
        df_delayed_sample = df.filter(F.col(target_col) == 1)
        df_us = df_ontime_sample.union(df_delayed_sample).cache()
    else:
        df_us = df
        print("  Already balanced — skipping undersampling.")
        return df_us

    n_us = df_us.count()
    n_us_del = df_us.filter(F.col(target_col) == 1).count()
    n_us_ot  = n_us - n_us_del
    print(f"  After:  on-time={n_us_ot:,}  delayed={n_us_del:,}  ratio={n_us_ot/max(n_us_del,1):.2f}:1")
    print(f"  Total:  {n_us:,}")
    return df_us


print("Undersampling StandardScaler train (for MLP):")
df_train_std_us = undersample(df_train_std, TARGET, UNDERSAMPLE_RATIO)

print("\nUndersampling PCA train (for MLP-PCA):")
df_train_pca_us = undersample(df_train_pca, TARGET, UNDERSAMPLE_RATIO)

Undersampling StandardScaler train (for MLP):
  Before: on-time=15,069,828  delayed=3,629,171  ratio=4.15:1
  After:  on-time=7,258,063  delayed=3,629,171  ratio=2.00:1
  Total:  10,887,234

Undersampling PCA train (for MLP-PCA):
  Before: on-time=15,069,828  delayed=3,629,171  ratio=4.15:1
  After:  on-time=7,258,063  delayed=3,629,171  ratio=2.00:1
  Total:  10,887,234


## 4. Evaluation Helpers — F-β Score (β=2)

All model comparisons use **F-β with β=2** as the primary metric, consistent with the
LR/RF/GBT notebooks. With β=2, recall is weighted 4× over precision:

$$F_2 = \frac{5 \cdot \text{Precision} \cdot \text{Recall}}{4 \cdot \text{Precision} + \text{Recall}}$$

In [0]:
def fbeta(precision, recall, beta=BETA):
    """Compute F-β score."""
    denom = beta**2 * precision + recall
    if denom < 1e-12:
        return 0.0
    return (1 + beta**2) * precision * recall / denom


def evaluate_model(model, df, label_col=TARGET, threshold=None):
    """
    Evaluate a fitted PySpark ML model.
    If threshold is not None, override the default 0.5 classification threshold.
    Returns a dict of metrics including F-β (primary), F1, AUROC, and confusion matrix.
    """
    predictions = model.transform(df)

    if threshold is not None:
        predictions = predictions.withColumn(
            "prediction",
            F.when(vector_to_array(F.col("probability"))[1] >= threshold, 1.0).otherwise(0.0),
        )

    auroc = BinaryClassificationEvaluator(
        rawPredictionCol="rawPrediction", labelCol=label_col, metricName="areaUnderROC"
    ).evaluate(predictions)

    tp = predictions.filter((F.col("prediction") == 1) & (F.col(label_col) == 1)).count()
    fp = predictions.filter((F.col("prediction") == 1) & (F.col(label_col) == 0)).count()
    fn = predictions.filter((F.col("prediction") == 0) & (F.col(label_col) == 1)).count()
    tn = predictions.filter((F.col("prediction") == 0) & (F.col(label_col) == 0)).count()

    prec_del = tp / max(tp + fp, 1)
    rec_del  = tp / max(tp + fn, 1)
    f1_del   = 2 * prec_del * rec_del / max(prec_del + rec_del, 1e-12)
    fb_del   = fbeta(prec_del, rec_del)

    accuracy = (tp + tn) / max(tp + fp + fn + tn, 1)

    return {
        f"F{BETA:.0f} (delayed)": round(fb_del, 4),
        "F1 (delayed)": round(f1_del, 4),
        "Precision (delayed)": round(prec_del, 4),
        "Recall (delayed)": round(rec_del, 4),
        "AUROC": round(auroc, 4),
        "Accuracy": round(accuracy, 4),
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
    }


def print_metrics(metrics, title=""):
    if title:
        print(f"\n{'='*60}")
        print(f"  {title}")
        print(f"{'='*60}")
    for k, v in metrics.items():
        if k in ("TP", "FP", "FN", "TN"):
            print(f"  {k:25s}: {v:>12,}")
        else:
            marker = "  <-- PRIMARY" if f"F{BETA:.0f}" in k and "delayed" in k else ""
            print(f"  {k:25s}: {v:>12.4f}{marker}")


def time_series_cv(df, estimator_fn, n_folds=3, label_col=TARGET, window_frac=0.20,
                   temporal_col="_temporal_sort"):
    """
    Rolling-window time-series cross-validation on training data.

    Uses a FIXED-SIZE training window that slides forward in time, ordered by
    a temporal column (YEAR*10000 + MONTH*100 + DAY_OF_MONTH). This ensures
    folds respect chronological order — earlier dates in train, later in val.

    Fold layout with n_folds=3, window_frac=0.20:
      Fold 1: Train [0%–20%]   → Val [20%–47%]
      Fold 2: Train [27%–47%]  → Val [47%–73%]
      Fold 3: Train [53%–73%]  → Val [73%–100%]
    """
    if temporal_col not in df.columns:
        raise ValueError(f"Temporal column '{temporal_col}' not found. "
                         f"Available: {df.columns}")

    df_sorted = df.cache()
    total = df_sorted.count()
    fold_metrics = []

    val_frac = (1.0 - window_frac) / n_folds

    for fold in range(n_folds):
        val_start  = window_frac + fold * val_frac
        val_end    = val_start + val_frac
        train_start = val_start - window_frac

        quantiles = df_sorted.approxQuantile(
            temporal_col,
            [train_start, val_start, val_end],
            0.01
        )
        train_lo, train_hi, val_hi = quantiles

        fold_train = df_sorted.filter(
            (F.col(temporal_col) >= train_lo) & (F.col(temporal_col) < train_hi)
        ).drop(temporal_col)

        fold_val = df_sorted.filter(
            (F.col(temporal_col) >= train_hi) & (F.col(temporal_col) < val_hi)
        ).drop(temporal_col)

        n_ft, n_fv = fold_train.count(), fold_val.count()
        if n_fv == 0 or n_ft == 0:
            continue

        # Apply undersampling within each fold's train split
        fold_train_us = undersample(fold_train, label_col, UNDERSAMPLE_RATIO, seed=SEED + fold)

        print(f"  Fold {fold+1}: Train {n_ft:,} → undersampled {fold_train_us.count():,} "
              f"| Val {n_fv:,}")
        model   = estimator_fn().fit(fold_train_us)
        metrics = evaluate_model(model, fold_val, label_col)
        metrics["fold"] = fold + 1
        fold_metrics.append(metrics)
        print(f"    F{BETA:.0f}(delayed)={metrics[f'F{BETA:.0f} (delayed)']:.4f}  "
              f"F1={metrics['F1 (delayed)']:.4f}  AUROC={metrics['AUROC']:.4f}")

    return fold_metrics


print("Evaluation helpers defined.")

Evaluation helpers defined.


## 5. MLP Architecture A: Single Hidden Layer `[N, 64, 2]`

The simplest architecture — one hidden layer with 64 units. Fast to train, serves as
a baseline for comparing deeper networks. Uses `l-bfgs` solver (quasi-Newton), which
is generally more stable than SGD for PySpark MLP on structured data.

In [0]:
LAYERS_A = [N_FEATURES, 64, 2]
print(f"Architecture A: {LAYERS_A}")

mlp_a = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=TARGET,
    layers=LAYERS_A,
    maxIter=200,
    blockSize=128,
    solver="l-bfgs",
    seed=SEED,
)

print("Training MLP-A on undersampled StandardScaler train...")
t0 = time.time()
mlp_a_model = mlp_a.fit(df_train_std_us)
mlp_a_time = time.time() - t0
print(f"MLP-A training done in {mlp_a_time:.1f}s")

mlp_a_val = evaluate_model(mlp_a_model, df_val_std)
print_metrics(mlp_a_val, "MLP-A [N,64,2] — VALIDATION")

Architecture A: [104, 64, 2]
Training MLP-A on undersampled StandardScaler train...
MLP-A training done in 7209.6s

  MLP-A [N,64,2] — VALIDATION
  F2 (delayed)             :       0.4554  <-- PRIMARY
  F1 (delayed)             :       0.3498
  Precision (delayed)      :       0.2523
  Recall (delayed)         :       0.5701
  AUROC                    :       0.6553
  Accuracy                 :       0.6612
  TP                       :      576,885
  FP                       :    1,709,493
  FN                       :      435,009
  TN                       :    3,608,151


## 6. MLP Architecture B: Two Hidden Layers `[N, 128, 64, 2]`

Deeper architecture with two hidden layers (128 → 64 units). The wider first layer
provides more capacity to learn from the 99 features before the 64-unit bottleneck
compresses the representation.

In [0]:
LAYERS_B = [N_FEATURES, 128, 64, 2]
print(f"Architecture B: {LAYERS_B}")

mlp_b = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=TARGET,
    layers=LAYERS_B,
    maxIter=200,
    blockSize=128,
    solver="l-bfgs",
    seed=SEED,
)

print("Training MLP-B on undersampled StandardScaler train...")
t0 = time.time()
mlp_b_model = mlp_b.fit(df_train_std_us)
mlp_b_time = time.time() - t0
print(f"MLP-B training done in {mlp_b_time:.1f}s")

mlp_b_val = evaluate_model(mlp_b_model, df_val_std)
print_metrics(mlp_b_val, "MLP-B [N,128,64,2] — VALIDATION")

Architecture B: [104, 128, 64, 2]
Training MLP-B on undersampled StandardScaler train...
MLP-B training done in 10076.6s

  MLP-B [N,128,64,2] — VALIDATION
  F2 (delayed)             :       0.4097  <-- PRIMARY
  F1 (delayed)             :       0.3501
  Precision (delayed)      :       0.2818
  Recall (delayed)         :       0.4622
  AUROC                    :       0.6784
  Accuracy                 :       0.7257
  TP                       :      467,689
  FP                       :    1,192,094
  FN                       :      544,205
  TN                       :    4,125,550


## 7. MLP Architecture C: Wider Network `[N, 256, 128, 2]`

Widest architecture — 256-unit first hidden layer for maximum representational capacity.
More parameters to tune, potentially slower to converge, but can capture richer
feature interactions.

In [0]:
LAYERS_C = [N_FEATURES, 256, 128, 2]
print(f"Architecture C: {LAYERS_C}")

mlp_c = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=TARGET,
    layers=LAYERS_C,
    maxIter=200,
    blockSize=128,
    solver="l-bfgs",
    seed=SEED,
)

print("Training MLP-C on undersampled StandardScaler train...")
t0 = time.time()
mlp_c_model = mlp_c.fit(df_train_std_us)
mlp_c_time = time.time() - t0
print(f"MLP-C training done in {mlp_c_time:.1f}s")

mlp_c_val = evaluate_model(mlp_c_model, df_val_std)
print_metrics(mlp_c_val, "MLP-C [N,256,128,2] — VALIDATION")

Architecture C: [104, 256, 128, 2]
Training MLP-C on undersampled StandardScaler train...
MLP-C training done in 21634.2s

  MLP-C [N,256,128,2] — VALIDATION
  F2 (delayed)             :       0.4365  <-- PRIMARY
  F1 (delayed)             :       0.3515
  Precision (delayed)      :       0.2654
  Recall (delayed)         :       0.5204
  AUROC                    :       0.6755
  Accuracy                 :       0.6930
  TP                       :      526,631
  FP                       :    1,458,016
  FN                       :      485,263
  TN                       :    3,859,628


## 8. Architecture Comparison

Compare all three architectures on validation set to select the best for further
experiments (threshold tuning, time-series CV, PCA comparison).

In [0]:
arch_results = {
    f"MLP-A {LAYERS_A}": {"metrics": mlp_a_val, "time": mlp_a_time, "model": mlp_a_model, "layers": LAYERS_A},
    f"MLP-B {LAYERS_B}": {"metrics": mlp_b_val, "time": mlp_b_time, "model": mlp_b_model, "layers": LAYERS_B},
    f"MLP-C {LAYERS_C}": {"metrics": mlp_c_val, "time": mlp_c_time, "model": mlp_c_model, "layers": LAYERS_C},
}

print(f"\n{'='*80}")
print(f"  ARCHITECTURE COMPARISON — VALIDATION (StandardScaler, undersampled 2:1)")
print(f"{'='*80}")
print(f"  {'Architecture':<25s} {'F2':>8s} {'F1':>8s} {'Prec':>8s} {'Rec':>8s} {'AUROC':>8s} {'Time(s)':>8s}")
print(f"  {'-'*73}")

best_arch_name = None
best_f2 = -1.0

for name, data in arch_results.items():
    m = data["metrics"]
    f2_key = f"F{BETA:.0f} (delayed)"
    row_f2 = m[f2_key]
    print(f"  {name:<25s} {row_f2:>8.4f} {m['F1 (delayed)']:>8.4f} "
          f"{m['Precision (delayed)']:>8.4f} {m['Recall (delayed)']:>8.4f} "
          f"{m['AUROC']:>8.4f} {data['time']:>8.1f}")
    if row_f2 > best_f2:
        best_f2 = row_f2
        best_arch_name = name

print(f"\n  Best architecture: {best_arch_name} (F2={best_f2:.4f})")

best_model = arch_results[best_arch_name]["model"]
best_time  = arch_results[best_arch_name]["time"]


  ARCHITECTURE COMPARISON — VALIDATION (StandardScaler, undersampled 2:1)
  Architecture                    F2       F1     Prec      Rec    AUROC  Time(s)
  -------------------------------------------------------------------------
  MLP-A [104, 64, 2]          0.4554   0.3498   0.2523   0.5701   0.6553   7209.6
  MLP-B [104, 128, 64, 2]     0.4097   0.3501   0.2818   0.4622   0.6784  10076.6
  MLP-C [104, 256, 128, 2]    0.4365   0.3515   0.2654   0.5204   0.6755  21634.2

  Best architecture: MLP-A [104, 64, 2] (F2=0.4554)


In [0]:
# Save architecture comparison chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

arch_names = list(arch_results.keys())
f2_key = f"F{BETA:.0f} (delayed)"
f2_vals = [arch_results[n]["metrics"][f2_key] for n in arch_names]
auroc_vals = [arch_results[n]["metrics"]["AUROC"] for n in arch_names]
train_times = [arch_results[n]["time"] for n in arch_names]

short_names = ["A: [N,64,2]", "B: [N,128,64,2]", "C: [N,256,128,2]"]

# F2 comparison
colors = ["#2196F3", "#FF9800", "#4CAF50"]
bars = axes[0].bar(short_names, f2_vals, color=colors, edgecolor="black", linewidth=0.5)
axes[0].set_ylabel("F2 Score (β=2)", fontsize=12)
axes[0].set_title("MLP Architecture Comparison — F2", fontsize=13)
axes[0].set_ylim(min(f2_vals) * 0.9, max(f2_vals) * 1.05)
for bar, val in zip(bars, f2_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

# AUROC comparison
bars2 = axes[1].bar(short_names, auroc_vals, color=colors, edgecolor="black", linewidth=0.5)
axes[1].set_ylabel("AUROC", fontsize=12)
axes[1].set_title("MLP Architecture Comparison — AUROC", fontsize=13)
axes[1].set_ylim(min(auroc_vals) * 0.95, max(auroc_vals) * 1.02)
for bar, val in zip(bars2, auroc_vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                 f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

fig.suptitle("MLP Neural Network — Architecture Comparison (Validation)", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/mlp_architecture_comparison.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {CHART_DIR}/mlp_architecture_comparison.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/mlp_architecture_comparison.png


## 9. PCA Experiment

Train the best-performing architecture on PCA-reduced features. This tests whether
dimensionality reduction improves MLP performance by removing multicollinearity and
noise dimensions.

In [0]:
# Determine layers for PCA version (input size changes)
best_layers_original = arch_results[best_arch_name]["layers"]

# Replace input dimension with PCA component count
pca_layers = [k_actual] + best_layers_original[1:]
print(f"PCA experiment: {best_arch_name}")
print(f"  Original layers: {best_layers_original}")
print(f"  PCA layers:      {pca_layers} (input reduced from {N_FEATURES} to {k_actual})")

mlp_pca = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol=TARGET,
    layers=pca_layers,
    maxIter=200,
    blockSize=128,
    solver="l-bfgs",
    seed=SEED,
)

print(f"\nTraining MLP-PCA on undersampled PCA train ({k_actual} components)...")
t0 = time.time()
mlp_pca_model = mlp_pca.fit(df_train_pca_us)
mlp_pca_time = time.time() - t0
print(f"MLP-PCA training done in {mlp_pca_time:.1f}s")

mlp_pca_val = evaluate_model(mlp_pca_model, df_val_pca)
print_metrics(mlp_pca_val, f"MLP-PCA {pca_layers} — VALIDATION")

# Compare with best StandardScaler model
f2_key = f"F{BETA:.0f} (delayed)"
print(f"\n  StandardScaler best: F2={best_f2:.4f}")
print(f"  PCA ({k_actual} comps):    F2={mlp_pca_val[f2_key]:.4f}")
delta = mlp_pca_val[f2_key] - best_f2
print(f"  Delta:               {'+'if delta>=0 else ''}{delta:.4f}")

PCA experiment: MLP-A [104, 64, 2]
  Original layers: [104, 64, 2]
  PCA layers:      [55, 64, 2] (input reduced from 104 to 55)

Training MLP-PCA on undersampled PCA train (55 components)...
MLP-PCA training done in 3445.3s

  MLP-PCA [55, 64, 2] — VALIDATION
  F2 (delayed)             :       0.3960  <-- PRIMARY
  F1 (delayed)             :       0.3446
  Precision (delayed)      :       0.2833
  Recall (delayed)         :       0.4397
  AUROC                    :       0.6282
  Accuracy                 :       0.7326
  TP                       :      444,921
  FP                       :    1,125,437
  FN                       :      566,973
  TN                       :    4,192,207

  StandardScaler best: F2=0.4554
  PCA (55 comps):    F2=0.3960
  Delta:               -0.0594


## 10. Threshold Tuning

MLP (like LR) defaults to a 0.5 decision threshold, which assumes symmetric
misclassification costs. With F2 (β=2) as our primary metric, we favor recall
over precision — a lower threshold biases toward predicting delays (fewer missed
delays, more false alarms).

We sweep thresholds on the **validation set** to find the threshold maximizing F2.

In [0]:
# Use the overall best model (StandardScaler or PCA, whichever is better)
if mlp_pca_val[f2_key] > best_f2:
    print("PCA model is best — using it for threshold tuning.")
    final_model = mlp_pca_model
    final_val_df = df_val_pca
    final_test_df = df_test_pca
    final_label = f"MLP-PCA {pca_layers}"
    final_preprocessing = "StandardScaler + PCA"
    best_time = mlp_pca_time
else:
    print("StandardScaler model is best — using it for threshold tuning.")
    final_model = best_model
    final_val_df = df_val_std
    final_test_df = df_test_std
    final_label = best_arch_name
    final_preprocessing = "StandardScaler"

StandardScaler model is best — using it for threshold tuning.


In [0]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]
threshold_results = []

print(f"\n{'='*80}")
print(f"  THRESHOLD TUNING — {final_label}")
print(f"{'='*80}")
print(f"  {'Thresh':>7s} {'Prec':>8s} {'Rec':>8s} {'F1':>8s} {'F2':>8s} {'AUROC':>8s}")
print(f"  {'-'*50}")

best_thresh = 0.5
best_thresh_f2 = -1.0

for thresh in thresholds:
    m = evaluate_model(final_model, final_val_df, threshold=thresh)
    m["threshold"] = thresh
    threshold_results.append(m)

    f2_val = m[f2_key]
    print(f"  {thresh:>7.2f} {m['Precision (delayed)']:>8.4f} {m['Recall (delayed)']:>8.4f} "
          f"{m['F1 (delayed)']:>8.4f} {f2_val:>8.4f} {m['AUROC']:>8.4f}"
          f"{'  <-- best' if f2_val > best_thresh_f2 else ''}")

    if f2_val > best_thresh_f2:
        best_thresh_f2 = f2_val
        best_thresh = thresh

print(f"\n  Best threshold: {best_thresh} (F2={best_thresh_f2:.4f})")


  THRESHOLD TUNING — MLP-A [104, 64, 2]
   Thresh     Prec      Rec       F1       F2    AUROC
  --------------------------------------------------
     0.10   0.1611   0.9977   0.2774   0.4894   0.6553  <-- best
     0.15   0.1659   0.9843   0.2840   0.4955   0.6553  <-- best
     0.20   0.1745   0.9548   0.2951   0.5041   0.6553  <-- best
     0.25   0.1848   0.9122   0.3074   0.5105   0.6553  <-- best
     0.30   0.1959   0.8614   0.3192   0.5128   0.6553  <-- best
     0.35   0.2074   0.8027   0.3297   0.5100   0.6553
     0.40   0.2203   0.7359   0.3391   0.5012   0.6553
     0.45   0.2350   0.6584   0.3464   0.4840   0.6553
     0.50   0.2523   0.5701   0.3498   0.4554   0.6553

  Best threshold: 0.3 (F2=0.5128)


In [0]:
# Save threshold tuning chart
fig, ax1 = plt.subplots(figsize=(10, 6))

thresh_vals = [r["threshold"] for r in threshold_results]
prec_vals = [r["Precision (delayed)"] for r in threshold_results]
rec_vals = [r["Recall (delayed)"] for r in threshold_results]
f2_thresh_vals = [r[f2_key] for r in threshold_results]
f1_thresh_vals = [r["F1 (delayed)"] for r in threshold_results]

ax1.plot(thresh_vals, prec_vals, "s-", color="#E91E63", label="Precision", linewidth=2, markersize=6)
ax1.plot(thresh_vals, rec_vals, "o-", color="#2196F3", label="Recall", linewidth=2, markersize=6)
ax1.plot(thresh_vals, f1_thresh_vals, "^-", color="#FF9800", label="F1", linewidth=2, markersize=6)
ax1.plot(thresh_vals, f2_thresh_vals, "D-", color="#4CAF50", label="F2 (primary)", linewidth=2.5, markersize=7)
ax1.axvline(x=best_thresh, color="gray", linestyle="--", alpha=0.7, label=f"Best threshold={best_thresh}")

ax1.set_xlabel("Decision Threshold", fontsize=12)
ax1.set_ylabel("Score", fontsize=12)
ax1.set_title(f"MLP Threshold Tuning — {final_label}", fontsize=14)
ax1.legend(fontsize=10, loc="center left")
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0.08, 0.52)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/mlp_threshold_tuning.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/mlp_threshold_tuning.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/mlp_threshold_tuning.png


## 11. Learning Curve Analysis (Pseudo-Early Stopping)

PySpark's MLP does not expose per-epoch training loss or provide an early-stopping
callback. To approximate training curves and analyze convergence, we train the best
architecture with **increasing `maxIter` values** and record validation F2 at each
checkpoint. This reveals:
- How quickly the model converges
- Whether additional iterations improve or degrade performance (overfitting signal)
- The "early stopping" point where validation performance plateaus

In [0]:
iter_checkpoints = [50, 100, 150, 200, 300]
learning_curve = []

# Determine layers for learning curve (use the final model's layers)
if final_preprocessing == "StandardScaler + PCA":
    lc_layers = pca_layers
    lc_train = df_train_pca_us
    lc_val = df_val_pca
else:
    lc_layers = best_layers_original
    lc_train = df_train_std_us
    lc_val = df_val_std

print(f"\n{'='*60}")
print(f"  LEARNING CURVE — {final_label}")
print(f"  Layers: {lc_layers}")
print(f"{'='*60}")
print(f"  {'maxIter':>8s} {'F2 (val)':>10s} {'AUROC':>8s} {'Time(s)':>8s}")
print(f"  {'-'*40}")

for max_iter in iter_checkpoints:
    mlp_lc = MultilayerPerceptronClassifier(
        featuresCol="features",
        labelCol=TARGET,
        layers=lc_layers,
        maxIter=max_iter,
        blockSize=128,
        solver="l-bfgs",
        seed=SEED,
    )

    t0 = time.time()
    lc_model = mlp_lc.fit(lc_train)
    lc_time = time.time() - t0

    lc_metrics = evaluate_model(lc_model, lc_val)
    lc_f2 = lc_metrics[f2_key]
    learning_curve.append({
        "maxIter": max_iter,
        "F2": lc_f2,
        "AUROC": lc_metrics["AUROC"],
        "time": lc_time,
    })
    print(f"  {max_iter:>8d} {lc_f2:>10.4f} {lc_metrics['AUROC']:>8.4f} {lc_time:>8.1f}")


  LEARNING CURVE — MLP-A [104, 64, 2]
  Layers: [104, 64, 2]
   maxIter   F2 (val)    AUROC  Time(s)
  ----------------------------------------
        50     0.4081   0.6648   1143.4
       100     0.4462   0.6610   2148.5
       150     0.4486   0.6551   3098.0
       200     0.4515   0.6532   4218.8
       300     0.4588   0.6536   6904.7


In [0]:
# Save learning curve chart
fig, ax1 = plt.subplots(figsize=(10, 6))

iters = [r["maxIter"] for r in learning_curve]
f2_lc = [r["F2"] for r in learning_curve]
auroc_lc = [r["AUROC"] for r in learning_curve]

ax1.plot(iters, f2_lc, "D-", color="#4CAF50", label="F2 (primary)", linewidth=2.5, markersize=8)
ax1.set_xlabel("maxIter (training iterations)", fontsize=12)
ax1.set_ylabel("F2 Score", fontsize=12, color="#4CAF50")
ax1.tick_params(axis="y", labelcolor="#4CAF50")

ax2 = ax1.twinx()
ax2.plot(iters, auroc_lc, "o--", color="#2196F3", label="AUROC", linewidth=2, markersize=7)
ax2.set_ylabel("AUROC", fontsize=12, color="#2196F3")
ax2.tick_params(axis="y", labelcolor="#2196F3")

ax1.set_title(f"MLP Learning Curve — {final_label}", fontsize=14)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc="lower right")
ax1.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/mlp_learning_curve.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/mlp_learning_curve.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/mlp_learning_curve.png


## 12. Time-Series Cross-Validation

Rolling-window time-series CV (3 folds) on the best MLP architecture, with inline
undersampling per fold. This validates that the model's performance is consistent
across different temporal windows.

In [0]:
# Use the full (non-undersampled) training set for CV — undersampling happens inside
if final_preprocessing == "StandardScaler + PCA":
    cv_train = df_train_pca
    cv_layers = pca_layers
else:
    cv_train = df_train_std
    cv_layers = best_layers_original

# Find best maxIter from learning curve
best_lc = max(learning_curve, key=lambda x: x["F2"])
best_max_iter = best_lc["maxIter"]
print(f"Using maxIter={best_max_iter} (best from learning curve, F2={best_lc['F2']:.4f})")

def mlp_estimator_fn():
    return MultilayerPerceptronClassifier(
        featuresCol="features",
        labelCol=TARGET,
        layers=cv_layers,
        maxIter=best_max_iter,
        blockSize=128,
        solver="l-bfgs",
        seed=SEED,
    )

print(f"\nTime-Series CV (3-fold rolling window) — {final_label}")
cv_metrics = time_series_cv(cv_train, mlp_estimator_fn, n_folds=3)

if cv_metrics:
    mean_f2   = np.mean([m[f2_key] for m in cv_metrics])
    std_f2    = np.std([m[f2_key] for m in cv_metrics])
    mean_auroc = np.mean([m["AUROC"] for m in cv_metrics])
    std_auroc  = np.std([m["AUROC"] for m in cv_metrics])
    print(f"\n  CV Summary:")
    print(f"    F2:    {mean_f2:.4f} ± {std_f2:.4f}")
    print(f"    AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")

Using maxIter=300 (best from learning curve, F2=0.4588)

Time-Series CV (3-fold rolling window) — MLP-A [104, 64, 2]
  Before: on-time=2,915,899  delayed=732,555  ratio=3.98:1
  After:  on-time=1,464,535  delayed=732,555  ratio=2.00:1
  Total:  2,197,090
  Fold 1: Train 3,648,454 → undersampled 2,197,090 | Val 4,967,662
    F2(delayed)=0.1647  F1=0.2127  AUROC=0.6419
  Before: on-time=2,934,564  delayed=677,829  ratio=4.33:1
  After:  on-time=1,355,172  delayed=677,829  ratio=2.00:1
  Total:  2,033,001
  Fold 2: Train 3,612,393 → undersampled 2,033,001 | Val 4,997,966
    F2(delayed)=0.4581  F1=0.3966  AUROC=0.6367
  Before: on-time=3,010,893  delayed=693,094  ratio=4.34:1
  After:  on-time=1,385,936  delayed=693,094  ratio=2.00:1
  Total:  2,079,030
  Fold 3: Train 3,703,987 → undersampled 2,079,030 | Val 5,063,313
    F2(delayed)=0.3127  F1=0.3430  AUROC=0.6662

  CV Summary:
    F2:    0.3118 ± 0.1198
    AUROC: 0.6483 ± 0.0129


## 13. Scikit-learn Ablation Study

PySpark's `MultilayerPerceptronClassifier` is limited: **no Adam optimizer** (only l-bfgs
and vanilla GD), **no regularization** (L1/L2), **no dropout**, and **no per-epoch loss
visibility**. To provide a rigorous neural network analysis, we complement the PySpark
experiments with scikit-learn's `MLPClassifier` on a stratified subsample.

| Capability | PySpark MLP | sklearn MLPClassifier |
|---|---|---|
| Optimizer | l-bfgs, GD | **Adam**, SGD, l-bfgs |
| Regularization | None | **L2 (alpha)** |
| Dropout | No | No |
| Training loss curve | Not exposed | **`loss_curve_`** |
| Early stopping | `tol` only | **Validation-based** |

**Why subsample?** sklearn runs on the driver node (single-machine). A stratified
~300K-row sample preserves class ratio and trains in seconds per config.

In [0]:
from sklearn.neural_network import MLPClassifier as SklearnMLP
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score

# ── 13a. Subsample Preparation ───────────────────────────────────────────────
# Convert PySpark features vector → numpy arrays on the driver

print("Preparing sklearn subsample from StandardScaler data...")
t0 = time.time()

# Undersampled train: take ~300K rows (or all if smaller)
us_count = df_train_std_us.count()
sample_frac = min(1.0, 300_000 / us_count)
pdf_train = (df_train_std_us
    .withColumn("features_array", vector_to_array("features"))
    .select("features_array", TARGET)
    .sample(fraction=sample_frac, seed=SEED)
    .toPandas())

X_train_sk = np.array(pdf_train["features_array"].tolist())
y_train_sk = pdf_train[TARGET].values.astype(int)

# Validation: take ~100K rows
val_count = df_val_std.count()
val_frac = min(1.0, 100_000 / val_count)
pdf_val = (df_val_std
    .withColumn("features_array", vector_to_array("features"))
    .select("features_array", TARGET)
    .sample(fraction=val_frac, seed=SEED)
    .toPandas())

X_val_sk = np.array(pdf_val["features_array"].tolist())
y_val_sk = pdf_val[TARGET].values.astype(int)

prep_time = time.time() - t0
print(f"  Train subsample: {len(X_train_sk):,} rows × {X_train_sk.shape[1]} features")
print(f"  Val subsample:   {len(X_val_sk):,} rows × {X_val_sk.shape[1]} features")
print(f"  Class balance (train): {y_train_sk.mean():.3f} delayed")
print(f"  Preparation time: {prep_time:.1f}s")

Preparing sklearn subsample from StandardScaler data...
  Train subsample: 299,980 rows × 104 features
  Val subsample:   100,211 rows × 104 features
  Class balance (train): 0.331 delayed
  Preparation time: 34.3s


In [0]:
def sklearn_fbeta(y_true, y_pred, beta=BETA):
    """Compute F-β from arrays."""
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    return fbeta(prec, rec, beta)


def eval_sklearn(clf, X, y, beta=BETA):
    """Evaluate an sklearn classifier and return metrics dict."""
    y_pred = clf.predict(X)
    y_prob = clf.predict_proba(X)[:, 1]
    prec = precision_score(y, y_pred, zero_division=0)
    rec = recall_score(y, y_pred, zero_division=0)
    return {
        f"F{beta:.0f}": round(fbeta(prec, rec, beta), 4),
        "F1": round(f1_score(y, y_pred, zero_division=0), 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "AUROC": round(roc_auc_score(y, y_prob), 4),
    }

### 13a. Training Loss Convergence

Train with **Adam optimizer** and early stopping, then plot the actual per-epoch
training loss via `loss_curve_`. This is the definitive proof that the model's
loss function converges — something PySpark MLP cannot provide.

In [0]:
# Match the best PySpark architecture's hidden layers
sk_hidden = tuple(best_layers_original[1:-1])  # e.g., (128, 64) from [N, 128, 64, 2]
print(f"sklearn architecture: hidden_layer_sizes={sk_hidden}")

clf_adam = SklearnMLP(
    hidden_layer_sizes=sk_hidden,
    solver="adam",
    alpha=0.0001,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=15,
    random_state=SEED,
    verbose=False,
)

print("Training sklearn MLP (Adam + early stopping + L2)...")
t0 = time.time()
clf_adam.fit(X_train_sk, y_train_sk)
sk_adam_time = time.time() - t0

n_epochs = len(clf_adam.loss_curve_)
print(f"  Converged after {n_epochs} epochs ({sk_adam_time:.1f}s)")
print(f"  Final training loss: {clf_adam.loss_curve_[-1]:.6f}")
print(f"  Best validation score: {clf_adam.best_validation_score_:.4f}")

sk_adam_val = eval_sklearn(clf_adam, X_val_sk, y_val_sk)
print(f"  Val F2: {sk_adam_val[f'F{BETA:.0f}']:.4f}  AUROC: {sk_adam_val['AUROC']:.4f}")

sklearn architecture: hidden_layer_sizes=(64,)
Training sklearn MLP (Adam + early stopping + L2)...
  Converged after 26 epochs (35.0s)
  Final training loss: 0.565651
  Best validation score: 0.7038
  Val F2: 0.4296  AUROC: 0.6754


In [0]:
# Save training loss convergence chart
fig, ax = plt.subplots(figsize=(10, 6))
epochs = range(1, len(clf_adam.loss_curve_) + 1)
ax.plot(epochs, clf_adam.loss_curve_, "b-", linewidth=2, label="Training Loss")
ax.axvline(x=n_epochs, color="r", linestyle="--", alpha=0.7,
           label=f"Stopped at epoch {n_epochs} (patience=15)")
ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss (Cross-Entropy)", fontsize=12)
ax.set_title(f"sklearn MLP Training Loss Convergence — Adam, hidden={sk_hidden}", fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/sklearn_loss_curve.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/sklearn_loss_curve.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/sklearn_loss_curve.png


### 13b. L2 Regularization Sweep

Sweep the L2 regularization strength (`alpha`) to find the optimal weight penalty.
Regularization prevents overfitting by penalizing large weights — it serves a
similar role to dropout (which neither PySpark nor sklearn MLP supports).

In [0]:
alphas = [0.0, 0.0001, 0.001, 0.01, 0.1]
reg_results = []

print(f"\n{'='*60}")
print(f"  L2 REGULARIZATION SWEEP — sklearn MLP (Adam)")
print(f"{'='*60}")
print(f"  {'alpha':>10s} {'F2 (val)':>10s} {'AUROC':>8s} {'Epochs':>8s} {'Time(s)':>8s}")
print(f"  {'-'*50}")

for alpha in alphas:
    clf_reg = SklearnMLP(
        hidden_layer_sizes=sk_hidden,
        solver="adam",
        alpha=alpha,
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=SEED,
        verbose=False,
    )
    t0 = time.time()
    clf_reg.fit(X_train_sk, y_train_sk)
    reg_time = time.time() - t0

    reg_metrics = eval_sklearn(clf_reg, X_val_sk, y_val_sk)
    n_ep = len(clf_reg.loss_curve_)
    reg_results.append({
        "alpha": alpha,
        **reg_metrics,
        "epochs": n_ep,
        "time": reg_time,
        "loss_curve": clf_reg.loss_curve_,
    })
    print(f"  {alpha:>10.4f} {reg_metrics[f'F{BETA:.0f}']:>10.4f} "
          f"{reg_metrics['AUROC']:>8.4f} {n_ep:>8d} {reg_time:>8.1f}")


  L2 REGULARIZATION SWEEP — sklearn MLP (Adam)
       alpha   F2 (val)    AUROC   Epochs  Time(s)
  --------------------------------------------------
      0.0000     0.4295   0.6757       26     32.7
      0.0001     0.4296   0.6754       26     33.1
      0.0010     0.4388   0.6725       38     47.7
      0.0100     0.4357   0.6744       38     59.3
      0.1000     0.3924   0.6795       26     61.8


In [0]:
# Save regularization sweep chart
fig, ax = plt.subplots(figsize=(10, 6))
alpha_labels = [f"{a:.4f}" if a > 0 else "0 (none)" for a in alphas]
f2_reg = [r[f"F{BETA:.0f}"] for r in reg_results]

bars = ax.bar(alpha_labels, f2_reg, color="#9C27B0", edgecolor="black", linewidth=0.5)
ax.set_xlabel("L2 Regularization Strength (α)", fontsize=12)
ax.set_ylabel("F2 Score (validation)", fontsize=12)
ax.set_title("L2 Regularization Sweep — sklearn MLP (Adam)", fontsize=14)

for bar, val in zip(bars, f2_reg):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"{val:.4f}", ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/sklearn_regularization_sweep.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/sklearn_regularization_sweep.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/sklearn_regularization_sweep.png


### 13c. Optimizer Comparison

Compare **Adam** (adaptive moment estimation), **l-bfgs** (quasi-Newton), and **SGD**
(stochastic gradient descent) on the same data. Adam is generally preferred for neural
networks because it adapts the learning rate per-parameter. PySpark MLP only supports
l-bfgs and vanilla GD — this experiment shows what we gain by using Adam.

In [0]:
# Find best alpha from regularization sweep
best_reg = max(reg_results, key=lambda x: x[f"F{BETA:.0f}"])
best_alpha = best_reg["alpha"]   
print(f"Using best alpha={best_alpha} from regularization sweep")
                                                                                                    
solvers = {                                                                                           
    "Adam": {"solver": "adam", "learning_rate_init": 0.001},
    "L-BFGS": {"solver": "lbfgs"},
    "SGD": {"solver": "sgd", "learning_rate_init": 0.001, "momentum": 0.9},
}
opt_results = []

print(f"\n{'='*60}")
print(f"  OPTIMIZER COMPARISON — sklearn MLP (alpha={best_alpha})")
print(f"{'='*60}")
print(f"  {'Solver':>10s} {'F2 (val)':>10s} {'AUROC':>8s} {'Epochs':>8s} {'Time(s)':>8s}")
print(f"  {'-'*50}")

for name, params in solvers.items():
    use_early_stopping = params.get("solver", "") != "lbfgs"
    clf_opt = SklearnMLP(
        hidden_layer_sizes=sk_hidden,
        alpha=best_alpha,
        max_iter=500,
        early_stopping=use_early_stopping,
        validation_fraction=0.15 if use_early_stopping else 0.0,
        n_iter_no_change=15 if use_early_stopping else 10,
        random_state=SEED,
        verbose=False,
        **params,
    )
    t0 = time.time()
    clf_opt.fit(X_train_sk, y_train_sk)
    opt_time = time.time() - t0

    opt_metrics = eval_sklearn(clf_opt, X_val_sk, y_val_sk)
    # loss_curve_ only exists for adam/sgd solvers, not lbfgs
    loss_curve = getattr(clf_opt, "loss_curve_", [])
    n_ep = len(loss_curve) if loss_curve else getattr(clf_opt, "n_iter_", 0)
    opt_results.append({
        "solver": name,
        **opt_metrics,
        "epochs": n_ep,
        "time": opt_time,
        "loss_curve": loss_curve,
    })
    print(f"  {name:>10s} {opt_metrics[f'F{BETA:.0f}']:>10.4f} "
        f"{opt_metrics['AUROC']:>8.4f} {n_ep:>8d} {opt_time:>8.1f}")

Using best alpha=0.001 from regularization sweep

  OPTIMIZER COMPARISON — sklearn MLP (alpha=0.001)
      Solver   F2 (val)    AUROC   Epochs  Time(s)
  --------------------------------------------------
        Adam     0.4388   0.6725       38     46.3
      L-BFGS     0.4444   0.6433      500    425.1
         SGD     0.4098   0.6732       71     80.0


In [0]:
# Save optimizer comparison chart — loss curves overlaid
fig, ax = plt.subplots(figsize=(10, 6))
colors_opt = {"Adam": "#4CAF50", "L-BFGS": "#2196F3", "SGD": "#FF9800"}

for r in opt_results:
    epochs = range(1, len(r["loss_curve"]) + 1)
    ax.plot(epochs, r["loss_curve"], linewidth=2, color=colors_opt[r["solver"]],
            label=f"{r['solver']} (F2={r[f'F{BETA:.0f}']:.4f}, {r['epochs']} epochs)")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Loss (Cross-Entropy)", fontsize=12)
ax.set_title(f"Optimizer Comparison — sklearn MLP, hidden={sk_hidden}", fontsize=13)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/sklearn_optimizer_comparison.png", dpi=150)
plt.close(fig)
print(f"Saved: {CHART_DIR}/sklearn_optimizer_comparison.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/sklearn_optimizer_comparison.png


### 13d. Feature Family Ablation

Systematically remove each feature family and measure the impact on F2. This
identifies which feature groups contribute most to the model's predictive power.

In [0]:
# Define feature families by column name
FEATURE_FAMILIES = {
    "Temporal": ["QUARTER", "MONTH", "DAY_OF_MONTH", "YEAR", "CRS_DEP_TIME", "dep_hour",
                 "is_morning_peak", "is_evening_peak", "is_weekend", "is_late_night",
                 "dep_hour_sin", "dep_hour_cos", "day_sin", "day_cos",
                 "month_sin", "month_cos", "quarter_sin", "quarter_cos"],
    "Holiday": ["is_holiday", "is_pre_holiday", "is_post_holiday", "is_holiday_weekend_adjacent"],
    "Congestion": ["origin_2h_sched_departures", "origin_sched_count_prev2h", "is_origin_busy_2h_before"],
    "Target Encoding": ["carrier_delay_rate", "origin_delay_rate", "dest_delay_rate",
                        "route_delay_rate", "hour_delay_rate"],
    "Weather": [c for c in numerical_cols if c.startswith("avg_Hourly") or c in
                ["precip_bin", "wind_bin", "vis_bin", "wx_evening_x_bad_vis", "wx_peak_x_strong_wind"]],
    "Airport Static": ["origin_train_departures", "dest_train_departures", "is_hub_origin",
                       "is_hub_dest", "origin_dest_distinct_count"],
    "Interactions": [c for c in numerical_cols if "_x_" in c and c not in
                     ["wx_evening_x_bad_vis", "wx_peak_x_strong_wind"]],
    "Indices": ["route_carrier_residual", "origin_pressure_index",
                "temp_anomaly_station_month", "severe_wx_index",
                "hour_month_cross_1", "hour_month_cross_2"],
}

# Map family → column indices in the feature vector
family_indices = {}
for family, cols in FEATURE_FAMILIES.items():
    indices = [numerical_cols.index(c) for c in cols if c in numerical_cols]
    family_indices[family] = indices
    print(f"  {family:<20s}: {len(indices)} features")

  Temporal            : 18 features
  Holiday             : 4 features
  Congestion          : 3 features
  Target Encoding     : 5 features
  Weather             : 20 features
  Airport Static      : 5 features
  Interactions        : 38 features
  Indices             : 6 features


In [0]:
# Baseline: all features
clf_baseline = SklearnMLP(
    hidden_layer_sizes=sk_hidden,
    solver="adam",
    alpha=best_alpha,
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=15,
    random_state=SEED,
    verbose=False,
)
clf_baseline.fit(X_train_sk, y_train_sk)
baseline_metrics = eval_sklearn(clf_baseline, X_val_sk, y_val_sk)
baseline_f2 = baseline_metrics[f"F{BETA:.0f}"]
print(f"Baseline (all features): F2={baseline_f2:.4f}")

# Ablation: remove each family
ablation_results = []

print(f"\n{'='*60}")
print(f"  FEATURE FAMILY ABLATION")
print(f"{'='*60}")
print(f"  {'Family Removed':<20s} {'F2':>8s} {'ΔF2':>8s} {'Features':>10s}")
print(f"  {'-'*50}")

for family, indices in family_indices.items():
    if not indices:
        continue

    # Create mask: keep all columns EXCEPT this family
    keep_mask = np.ones(X_train_sk.shape[1], dtype=bool)
    keep_mask[indices] = False

    X_train_abl = X_train_sk[:, keep_mask]
    X_val_abl = X_val_sk[:, keep_mask]

    # Adjust hidden layers for reduced input (keep same proportions)
    clf_abl = SklearnMLP(
        hidden_layer_sizes=sk_hidden,
        solver="adam",
        alpha=best_alpha,
        learning_rate_init=0.001,
        max_iter=500,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=15,
        random_state=SEED,
        verbose=False,
    )
    clf_abl.fit(X_train_abl, y_train_sk)
    abl_metrics = eval_sklearn(clf_abl, X_val_abl, y_val_sk)
    abl_f2 = abl_metrics[f"F{BETA:.0f}"]
    delta_f2 = abl_f2 - baseline_f2

    ablation_results.append({
        "family": family,
        "f2": abl_f2,
        "delta_f2": delta_f2,
        "n_removed": len(indices),
    })
    print(f"  {family:<20s} {abl_f2:>8.4f} {delta_f2:>+8.4f} {len(indices):>10d}")

Baseline (all features): F2=0.4388

  FEATURE FAMILY ABLATION
  Family Removed             F2      ΔF2   Features
  --------------------------------------------------
  Temporal               0.3175  -0.1213         18
  Holiday                0.4200  -0.0188          4
  Congestion             0.4328  -0.0060          3
  Target Encoding        0.3839  -0.0549          5
  Weather                0.4326  -0.0062         20
  Airport Static         0.3986  -0.0402          5
  Interactions           0.3712  -0.0676         38
  Indices                0.3615  -0.0773          6


In [0]:
# Save ablation chart
ablation_results.sort(key=lambda x: x["delta_f2"])

fig, ax = plt.subplots(figsize=(10, 6))
families = [r["family"] for r in ablation_results]
deltas = [r["delta_f2"] for r in ablation_results]
bar_colors = ["#F44336" if d < 0 else "#4CAF50" for d in deltas]

bars = ax.barh(families, deltas, color=bar_colors, edgecolor="black", linewidth=0.5)
ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_xlabel("ΔF2 (change from baseline when family is removed)", fontsize=12)
ax.set_title("Feature Family Ablation Study — Impact on F2", fontsize=14)

for bar, val in zip(bars, deltas):
    x_pos = bar.get_width() + (0.001 if val >= 0 else -0.001)
    ha = "left" if val >= 0 else "right"
    ax.text(x_pos, bar.get_y() + bar.get_height()/2,
            f"{val:+.4f}", va="center", ha=ha, fontsize=9, fontweight="bold")

ax.grid(True, axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{CHART_DIR}/feature_ablation.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {CHART_DIR}/feature_ablation.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/feature_ablation.png


### 13e. Sklearn Ablation Summary

The scikit-learn ablation study complements the PySpark full-scale experiments by
providing capabilities PySpark MLP lacks:

- **Convergence proof**: `loss_curve_` shows training loss decreasing monotonically
- **Adam optimizer**: Adaptive learning rates generally outperform l-bfgs and SGD
- **L2 regularization**: Optimal `alpha` prevents overfitting without dropout
- **Feature ablation**: Identifies which feature families drive predictions most

**Limitation**: Neither PySpark nor sklearn MLP supports **dropout**. For proper
dropout regularization, a PyTorch/TensorFlow implementation would be needed — this
is a natural future direction.

## 14. Blind Test Evaluation

Final evaluation on the held-out **2019 test set** (blind — never used during training
or threshold tuning). The decision threshold is **frozen** from validation tuning.

In [0]:
print(f"\n{'='*60}")
print(f"  BLIND TEST — {final_label}")
print(f"  Threshold: {best_thresh} (frozen from validation)")
print(f"{'='*60}")

final_test_metrics = evaluate_model(final_model, final_test_df, threshold=best_thresh)
print_metrics(final_test_metrics, f"{final_label} — TEST (BLIND)")

# Also evaluate at default threshold for comparison
final_test_default = evaluate_model(final_model, final_test_df)
print_metrics(final_test_default, f"{final_label} — TEST (threshold=0.5)")


  BLIND TEST — MLP-A [104, 64, 2]
  Threshold: 0.3 (frozen from validation)

  MLP-A [104, 64, 2] — TEST (BLIND)
  F2 (delayed)             :       0.5212  <-- PRIMARY
  F1 (delayed)             :       0.3123
  Precision (delayed)      :       0.1872
  Recall (delayed)         :       0.9406
  AUROC                    :       0.6505
  Accuracy                 :       0.3015
  TP                       :      989,930
  FP                       :    4,297,014
  FN                       :       62,546
  TN                       :      891,518

  MLP-A [104, 64, 2] — TEST (threshold=0.5)
  F2 (delayed)             :       0.5074  <-- PRIMARY
  F1 (delayed)             :       0.3488
  Precision (delayed)      :       0.2293
  Recall (delayed)         :       0.7280
  AUROC                    :       0.6505
  Accuracy                 :       0.5415
  TP                       :      766,225
  FP                       :    2,575,059
  FN                       :      286,251
  TN             

## 15. Results Summary & Comparison

Aggregate all MLP results and compare with existing LR, RF, GBT performance.

In [0]:
# ── Build MLP results table ──────────────────────────────────────────────────

results_rows = []

# All architectures on validation (default threshold)
for name, data in arch_results.items():
    m = data["metrics"]
    results_rows.append({
        "Model": f"{name} (StandardScaler)",
        "Split": "Validation",
        **{k: v for k, v in m.items()},
        "Train Time (s)": round(data["time"], 1),
    })

# PCA on validation
results_rows.append({
    "Model": f"MLP-PCA {pca_layers} (StdScaler+PCA)",
    "Split": "Validation",
    **{k: v for k, v in mlp_pca_val.items()},
    "Train Time (s)": round(mlp_pca_time, 1),
})

# Best model with tuned threshold on validation
best_thresh_val_metrics = evaluate_model(final_model, final_val_df, threshold=best_thresh)
results_rows.append({
    "Model": f"{final_label} (thresh={best_thresh})",
    "Split": "Validation",
    **{k: v for k, v in best_thresh_val_metrics.items()},
    "Train Time (s)": round(best_time, 1),
})

# Best model on blind test
results_rows.append({
    "Model": f"{final_label} (thresh={best_thresh})",
    "Split": "Test (Blind)",
    **{k: v for k, v in final_test_metrics.items()},
    "Train Time (s)": "",
})

results_df = pd.DataFrame(results_rows)
print("\nMLP Results Summary:")
print(results_df.to_string(index=False))


MLP Results Summary:
                                    Model        Split  F2 (delayed)  F1 (delayed)  Precision (delayed)  Recall (delayed)  AUROC  Accuracy     TP      FP     FN      TN Train Time (s)
      MLP-A [104, 64, 2] (StandardScaler)   Validation        0.4554        0.3498               0.2523            0.5701 0.6553    0.6612 576885 1709493 435009 3608151         7209.6
 MLP-B [104, 128, 64, 2] (StandardScaler)   Validation        0.4097        0.3501               0.2818            0.4622 0.6784    0.7257 467689 1192094 544205 4125550        10076.6
MLP-C [104, 256, 128, 2] (StandardScaler)   Validation        0.4365        0.3515               0.2654            0.5204 0.6755    0.6930 526631 1458016 485263 3859628        21634.2
      MLP-PCA [55, 64, 2] (StdScaler+PCA)   Validation        0.3960        0.3446               0.2833            0.4397 0.6282    0.7326 444921 1125437 566973 4192207         3445.3
          MLP-A [104, 64, 2] (thresh=0.3)   Validation    

In [0]:
# ── Save MLP results CSV to DBFS ─────────────────────────────────────────────

# Select columns matching the format of existing results CSVs
csv_cols = ["Model", "Split", f"F{BETA:.0f} (delayed)", "F1 (delayed)",
            "Precision (delayed)", "Recall (delayed)", "AUROC", "Accuracy",
            "TP", "FP", "FN", "TN", "Train Time (s)"]

results_csv_df = results_df[[c for c in csv_cols if c in results_df.columns]]
csv_path_local = "/tmp/phase3_results_summary_mlp.csv"
results_csv_df.to_csv(csv_path_local, index=False)

dbutils.fs.cp(f"file:{csv_path_local}", MLP_RESULTS_CSV)
print(f"Saved: {MLP_RESULTS_CSV}")

Saved: dbfs:/student-groups/Group_01_02/phase3_results_summary_mlp.csv


In [0]:
# ── Load and compare with existing model results ─────────────────────────────

print("\n" + "="*80)
print("  ALL MODELS COMPARISON")
print("="*80)

# Load existing results
existing_files = {
    "LR/RF": f"{BASE_GROUP}/phase3_results_summary_lr_rf.csv",
    "GBT":   f"{BASE_GROUP}/phase3_results_summary_gbt.csv",
}

all_results = []
for label, path in existing_files.items():
    try:
        csv_text = dbutils.fs.head(path, 100000)
        df_existing = pd.read_csv(StringIO(csv_text))
        all_results.append(df_existing)
        print(f"  Loaded {label} results: {len(df_existing)} rows")
    except Exception as e:
        print(f"  Could not load {label}: {e}")

all_results.append(results_csv_df)

if all_results:
    combined_df = pd.concat(all_results, ignore_index=True)

    # Filter to validation and test rows only
    for split_name in ["Validation", "Test (Blind)"]:
        split_df = combined_df[combined_df["Split"] == split_name]
        if not split_df.empty:
            print(f"\n  {split_name}:")
            print(f"  {'Model':<45s} {'F2':>7s} {'F1':>7s} {'Prec':>7s} {'Rec':>7s} {'AUROC':>7s}")
            print(f"  {'-'*80}")
            for _, row in split_df.iterrows():
                print(f"  {str(row['Model']):<45s} "
                      f"{row.get(f'F{BETA:.0f} (delayed)', 'N/A'):>7} "
                      f"{row.get('F1 (delayed)', 'N/A'):>7} "
                      f"{row.get('Precision (delayed)', 'N/A'):>7} "
                      f"{row.get('Recall (delayed)', 'N/A'):>7} "
                      f"{row.get('AUROC', 'N/A'):>7}")


  ALL MODELS COMPARISON
  Loaded LR/RF results: 4 rows
  Loaded GBT results: 2 rows

  Validation:
  Model                                              F2      F1    Prec     Rec   AUROC
  --------------------------------------------------------------------------------
  LR (grid search best)                          0.5077  0.3255  0.2037  0.8098  0.6624
  RF (undersampled, thresh=0.4)                  0.5147  0.3224  0.1987  0.8544  0.6741
  GBT (grid search, thresh=0.25)                 0.5182  0.3251  0.2005  0.8582  0.6828
  MLP-A [104, 64, 2] (StandardScaler)            0.4554  0.3498  0.2523  0.5701  0.6553
  MLP-B [104, 128, 64, 2] (StandardScaler)       0.4097  0.3501  0.2818  0.4622  0.6784
  MLP-C [104, 256, 128, 2] (StandardScaler)      0.4365  0.3515  0.2654  0.5204  0.6755
  MLP-PCA [55, 64, 2] (StdScaler+PCA)             0.396  0.3446  0.2833  0.4397  0.6282
  MLP-A [104, 64, 2] (thresh=0.3)                0.5128  0.3192  0.1959  0.8614  0.6553

  Test (Blind):
  Model 

In [0]:
# ── Save all-models comparison chart ─────────────────────────────────────────

if all_results:
    val_df = combined_df[combined_df["Split"] == "Validation"].copy()
    if not val_df.empty and f"F{BETA:.0f} (delayed)" in val_df.columns:
        val_df[f"F{BETA:.0f} (delayed)"] = pd.to_numeric(val_df[f"F{BETA:.0f} (delayed)"], errors="coerce")
        val_df = val_df.dropna(subset=[f"F{BETA:.0f} (delayed)"])
        val_df = val_df.sort_values(f"F{BETA:.0f} (delayed)", ascending=True)

        fig, ax = plt.subplots(figsize=(12, max(6, len(val_df) * 0.5)))
        y_pos = range(len(val_df))
        f2_col = f"F{BETA:.0f} (delayed)"

        # Color MLP bars differently
        bar_colors = ["#FF9800" if "MLP" in str(m) else "#2196F3" for m in val_df["Model"]]
        bars = ax.barh(y_pos, val_df[f2_col].values, color=bar_colors,
                       edgecolor="black", linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(val_df["Model"].values, fontsize=9)
        ax.set_xlabel("F2 Score (β=2)", fontsize=12)
        ax.set_title("All Models — Validation F2 Comparison", fontsize=14)

        for bar, val in zip(bars, val_df[f2_col].values):
            ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                    f"{val:.4f}", va="center", fontsize=9, fontweight="bold")

        ax.grid(True, axis="x", alpha=0.3)
        fig.tight_layout()
        fig.savefig(f"{CHART_DIR}/all_models_comparison.png", dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved: {CHART_DIR}/all_models_comparison.png")

Saved: /dbfs/FileStore/group_01_02/phase3_charts/all_models_comparison.png


## 16. Summary

This notebook trained **three MLP architectures** on StandardScaler-normalized features
and one additional PCA-reduced variant:

| Architecture | Layers | Preprocessing | Notes |
|---|---|---|---|
| MLP-A | `[N, 64, 2]` | StandardScaler | Shallow baseline |
| MLP-B | `[N, 128, 64, 2]` | StandardScaler | Two hidden layers |
| MLP-C | `[N, 256, 128, 2]` | StandardScaler | Wider first layer |
| MLP-PCA | `[k, *, 2]` | StandardScaler + PCA | Dimensionality reduction |

**Experiments conducted:**
- Architecture comparison (F2 primary metric)
- Threshold tuning (9 thresholds on validation)
- Learning curve analysis (5 maxIter checkpoints → pseudo-early-stopping)
- **Scikit-learn ablation study**: Adam optimizer, L2 regularization sweep,
  optimizer comparison, feature family ablation
- Time-series cross-validation (3-fold rolling window with inline undersampling)
- PCA variance analysis and dimensionality reduction experiment
- Blind test evaluation with frozen optimal threshold

**Results saved to:**
- `dbfs:/student-groups/Group_01_02/phase3_results_summary_mlp.csv`
- Charts: `dbfs:/FileStore/group_01_02/phase3_charts/`